In [1]:
import numpy as np
from tqdm import tqdm
import pickle
import os
import argparse

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import torch

import modeling as M
from src.utils import QFilters


device = "cuda" if torch.cuda.is_available() else "cpu"

/home/sh1ng/dev/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
model_cls = getattr(M, 'Qwen2ForCausalLM')
max_seq_len = 1024
num_sequences = 25
num_svd_samples = 5000
filter_suffix = ""
torch_dtype = 'bfloat16'

dataset_name = 'PatrickHaller/fineweb-1B'
dataset_config = 'default'
dataset_split = 'train[:1000]'


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = model_cls.from_pretrained(
    model_name, attn_implementation="flash_attention_2", device_map="auto",  low_cpu_mem_usage=True, torch_dtype=torch_dtype)

model = model.eval()

dataset = load_dataset(dataset_name, dataset_config, split=dataset_split)


with torch.no_grad():
    decoder = getattr(model, "gpt_neox", getattr(model, 'model', None))
    svd_filters = [[] for _ in range(len(decoder.layers))]
    sample_count = 0
    num_k_heads = None

    for i, sample in tqdm(enumerate(dataset)):

        tokens = tokenizer(sample["text"], return_tensors="pt")
        if tokens.input_ids.shape[-1] < max_seq_len:
            continue
        sample_count+=1
        input_ids = tokens.input_ids[:, :max_seq_len].to(device)
        if sample_count < num_sequences:
            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                out_repr = model(input_ids).past_key_values
            for j, (query, key) in enumerate(out_repr):
                num_k_heads = key.shape[1]
                svd_filters[j].append(query.flatten(0, 1).cpu())
        else:
            break

    del model

139it [00:05, 26.57it/s]


In [4]:
q_filters = [[] for _ in range(len(decoder.layers))]
vhs = []
for f_id, el in enumerate(svd_filters):
        stacked_el = torch.stack(el, 1).flatten(1, 2)
        idx = torch.argsort(torch.rand(stacked_el.shape[1], device=stacked_el.device))[:num_svd_samples]
        stacked_el = stacked_el[:, idx].cuda()
        u,s,vh = torch.linalg.svd(stacked_el.float(), full_matrices=False)
        vhs.append(vh)
        svd_sign = ((u[..., 0]>0).float().mean(-1) > 0.5).float()*2-1
        svd_filter_q = -svd_sign[:, None] * vh[..., 0, :]
        q_filters[f_id] = svd_filter_q.reshape(num_k_heads, -1, svd_filter_q.shape[-1]).mean(-2)
vhs = torch.stack(vhs, 0)

In [5]:
print(u.shape, s.shape, vh.shape)

torch.Size([12, 5000, 128]) torch.Size([12, 128]) torch.Size([12, 128, 128])


In [6]:
kv_cache = []
for idx, el in enumerate(svd_filters):
    kv_cache.append(torch.stack(el, 0))
kv_cache = torch.stack(kv_cache, 1)

In [11]:
dot = torch.einsum('slhci,lhji->slhcj', kv_cache.float(), vhs.cpu())

In [12]:
dot.shape

torch.Size([24, 28, 12, 1024, 128])

In [13]:
sums_abs = dot.sum(dim=3).abs()

In [14]:

np.bincount(sums_abs.argmax(3).flatten())

array([7872,   24,   72,    0,    0,    0,   48,    0,   24,    0,    0,
          0,    0,    0,   24])